# EmotionVision — Training on Kaggle (GPU T4)

**Before running:**
1. *Settings → Accelerator → GPU T4 x1* and *Internet → On*
2. *Add-ons → Secrets*: add `DAGSHUB_USER`, `DAGSHUB_TOKEN` (and `GITHUB_TOKEN` if the repo is private)
3. Edit `REPO_OWNER` below

In [ ]:
REPO_OWNER = "<your-github-username>"   # same owner on GitHub and DagsHub
REPO_NAME = "emotionvision"

from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
DAGSHUB_USER = s.get_secret("DAGSHUB_USER")
DAGSHUB_TOKEN = s.get_secret("DAGSHUB_TOKEN")
try:
    GITHUB_TOKEN = s.get_secret("GITHUB_TOKEN")
    clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
except Exception:
    clone_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

In [ ]:
%cd /kaggle/working
!rm -rf {REPO_NAME}
!git clone -q {clone_url}
%cd {REPO_NAME}
# torch/torchvision are preinstalled on Kaggle: only install what is missing
!pip install -q "dvc[s3]" "mlflow>=2.16,<3" dagshub timm

In [ ]:
# Pull the processed dataset (output of the DVC `prepare` stage) from DagsHub
!dvc remote modify origin --local access_key_id {DAGSHUB_TOKEN}
!dvc remote modify origin --local secret_access_key {DAGSHUB_TOKEN}
!dvc pull prepare
!ls data/processed/train

In [ ]:
import os
os.environ["MLFLOW_TRACKING_URI"] = f"https://dagshub.com/{REPO_OWNER}/{REPO_NAME}.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USER
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Experiment 1 — Baseline CNN from scratch

In [ ]:
!python -m src.training.train --model baseline_cnn --no-pretrained --epochs 40 --lr 0.001 --num-workers 4

## Experiment 2 — EfficientNet-B0 (ImageNet, 3 epochs head-only, then full fine-tuning)

In [ ]:
!python -m src.training.train --model efficientnet_b0 --num-workers 4

## Experiment 3 — ResNet-50 (ImageNet, same protocol)

In [ ]:
!python -m src.training.train --model resnet50 --batch-size 96 --num-workers 4

## Register the best run (chosen on **validation** macro-F1, not test) as `champion`

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()
exp = client.get_experiment_by_name("emotionvision")
runs = client.search_runs([exp.experiment_id], order_by=["metrics.best_val_macro_f1 DESC"], max_results=10)
for r in runs:
    m = r.data.metrics
    print(f"{r.data.params.get('model'):16s} val_f1={m.get('best_val_macro_f1', 0):.4f} "
          f"test_acc={m.get('test_accuracy', 0):.4f} test_f1={m.get('test_macro_f1', 0):.4f}  {r.info.run_id}")

best = runs[0]
mv = mlflow.register_model(f"runs:/{best.info.run_id}/model", "emotionvision")
try:
    client.set_registered_model_alias("emotionvision", "champion", mv.version)
except Exception as e:  # older registry without aliases
    client.set_model_version_tag("emotionvision", mv.version, "role", "champion")
print(f"Registered {best.data.params['model']} as emotionvision v{mv.version} (champion)")